In [8]:
from google.colab import userdata, drive
import os, subprocess, shutil, datetime

drive.mount('/content/drive', force_remount=True)

# ── EDIT THESE TWO LINES ONLY ──────────────────────────────
GITHUB_EMAIL = "sindhu.pl@outlook.com"   # your GitHub account email
# ───────────────────────────────────────────────────────────

GITHUB_USERNAME = "Lakshmi-Sindhu-P"
GITHUB_REPO     = "Campaign-Prophet"
GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')

DRIVE_ROOT  = "/content/drive/MyDrive/Campaign_Prophet"
LOCAL_REPO  = f"/content/{GITHUB_REPO}"
REPO_URL    = (f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}"
               f"@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git")

subprocess.run(['git', 'config', '--global', 'user.email', GITHUB_EMAIL], check=True)
subprocess.run(['git', 'config', '--global', 'user.name', GITHUB_USERNAME], check=True)

print("✓ Configuration complete")
print(f"  Drive root : {DRIVE_ROOT}")
print(f"  Repository : github.com/{GITHUB_USERNAME}/{GITHUB_REPO}")

Mounted at /content/drive
✓ Configuration complete
  Drive root : /content/drive/MyDrive/Campaign_Prophet
  Repository : github.com/Lakshmi-Sindhu-P/Campaign-Prophet


In [9]:
# Clone repo if first time, pull if already exists

if not os.path.exists(LOCAL_REPO):
    result = subprocess.run(
        ['git', 'clone', REPO_URL, LOCAL_REPO],
        capture_output=True, text=True
    )
    print("✓ Repository cloned")
    if result.stderr:
        print(result.stderr)
else:
    result = subprocess.run(
        ['git', '-C', LOCAL_REPO, 'pull'],
        capture_output=True, text=True
    )
    print("✓ Repository pulled")
    print(result.stdout.strip())

✓ Repository pulled
Updating ed3a2b4..f20d7d5
Fast-forward
 LICENSE   |  21 +++
 README.md | 564 +++++++++++++++++++++++++++++++++++---------------------------
 2 files changed, 339 insertions(+), 246 deletions(-)
 create mode 100644 LICENSE


In [10]:
# Define what to sync and what to skip

EXCLUDE_EXTENSIONS = {'.pkl', '.zip', '.pyc'}
EXCLUDE_FOLDERS    = {'.ipynb_checkpoints', '__pycache__', 'backup', '.git'}
EXCLUDE_FILES      = {'.DS_Store', 'Thumbs.db'}
MAX_FILE_MB        = 50

def should_sync(path):
    name = os.path.basename(path)
    if name in EXCLUDE_FILES:
        return False
    _, ext = os.path.splitext(name)
    if ext.lower() in EXCLUDE_EXTENSIONS:
        return False
    if os.path.isdir(path) and name in EXCLUDE_FOLDERS:
        return False
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        if size_mb > MAX_FILE_MB:
            print(f"  ⚠ Skipped (too large {size_mb:.1f}MB): {name}")
            return False
    return True

print("✓ Sync rules defined")
print(f"  Excluded extensions : {EXCLUDE_EXTENSIONS}")
print(f"  Excluded folders    : {EXCLUDE_FOLDERS}")
print(f"  Max file size       : {MAX_FILE_MB}MB")

✓ Sync rules defined
  Excluded extensions : {'.pyc', '.pkl', '.zip'}
  Excluded folders    : {'backup', '.git', '.ipynb_checkpoints', '__pycache__'}
  Max file size       : 50MB


In [11]:
# Mirror Drive folder structure to local repo

def sync_folder(src, dst):
    os.makedirs(dst, exist_ok=True)
    synced, skipped = [], []
    for item in sorted(os.listdir(src)):
        src_item = os.path.join(src, item)
        dst_item = os.path.join(dst, item)
        if not should_sync(src_item):
            skipped.append(item)
            continue
        if os.path.isdir(src_item):
            s, sk = sync_folder(src_item, dst_item)
            synced.extend(s)
            skipped.extend(sk)
        else:
            shutil.copy2(src_item, dst_item)
            synced.append(dst_item.replace(LOCAL_REPO, ''))
    return synced, skipped

print("Syncing Drive → local repo...\n")
synced, skipped = sync_folder(DRIVE_ROOT, LOCAL_REPO)
print(f"✓ Sync complete")
print(f"  Files synced  : {len(synced)}")
print(f"  Files skipped : {len(skipped)}")

if synced:
    print("\nSynced files:")
    for f in synced:
        print(f"  {f}")

Syncing Drive → local repo...

✓ Sync complete
  Files synced  : 17
  Files skipped : 8

Synced files:
  /README.md
  /data/README.md
  /data/processed/README.md
  /data/processed/cleaned_feature_engineered_bank_marketing.csv
  /data/raw/README.md
  /notebooks/00_github_sync.ipynb
  /notebooks/01_business_roi_statistical_validation.ipynb
  /notebooks/02_modeling_targeting_campaign_optimization.ipynb
  /outputs/notebook_01/age_summary.csv
  /outputs/notebook_01/cleaned_feature_engineered_bank_marketing.csv
  /outputs/notebook_01/duration_summary.csv
  /outputs/notebook_01/expected_value_eur_base.csv
  /outputs/notebook_01/expected_value_usd_base.csv
  /outputs/notebook_01/job_subscription_summary.csv
  /outputs/notebook_01/job_summary.csv
  /outputs/notebook_01/poutcome_subscription_summary.csv
  /outputs/notebook_01/usd_eur_ranking_comparison.csv


In [12]:
# Stage, commit, and push to GitHub

subprocess.run(['git', '-C', LOCAL_REPO, 'add', '.'])

status = subprocess.run(
    ['git', '-C', LOCAL_REPO, 'status', '--porcelain'],
    capture_output=True, text=True
)

if not status.stdout.strip():
    print("✓ Nothing to commit — repository is already up to date.")
else:
    # Show what changed
    print("Changes to be pushed:")
    for line in status.stdout.strip().split('\n'):
        print(f"  {line}")

    timestamp   = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    commit_msg  = f"Sync from Google Drive — {timestamp}"

    commit = subprocess.run(
        ['git', '-C', LOCAL_REPO, 'commit', '-m', commit_msg],
        capture_output=True, text=True
    )
    print(f"\n✓ Committed: {commit_msg}")

    push = subprocess.run(
        ['git', '-C', LOCAL_REPO, 'push'],
        capture_output=True, text=True
    )

    if push.returncode == 0:
        print(f"✓ Pushed successfully to GitHub")
        print(f"  View at: https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}")
    else:
        print("✗ Push failed:")
        print(push.stderr)

Changes to be pushed:
  M  README.md
  M  notebooks/00_github_sync.ipynb

✓ Committed: Sync from Google Drive — 2026-06-07 04:50
✓ Pushed successfully to GitHub
  View at: https://github.com/Lakshmi-Sindhu-P/Campaign-Prophet


In [13]:
# Show final repository structure

print("Repository structure on GitHub:\n")

for root, dirs, files in os.walk(LOCAL_REPO):
    dirs[:] = sorted([d for d in dirs if d not in
                      {'.git', '__pycache__', '.ipynb_checkpoints'}])
    level   = root.replace(LOCAL_REPO, '').count(os.sep)
    indent  = '  ' * level
    name    = os.path.basename(root) or GITHUB_REPO
    print(f"{indent}📁 {name}/")
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{'  ' * (level+1)}📄 {f}  ({size_kb:.0f}KB)")

Repository structure on GitHub:

📁 Campaign-Prophet/
  📄 LICENSE  (1KB)
  📄 README.md  (12KB)
  📁 data/
    📄 README.md  (0KB)
    📁 processed/
      📄 README.md  (0KB)
      📄 cleaned_feature_engineered_bank_marketing.csv  (5480KB)
    📁 raw/
      📄 README.md  (0KB)
  📁 docs/
  📁 notebooks/
    📄 00_github_sync.ipynb  (12KB)
    📄 01_business_roi_statistical_validation.ipynb  (363KB)
    📄 02_modeling_targeting_campaign_optimization.ipynb  (407KB)
  📁 outputs/
    📁 notebook_01/
      📄 age_summary.csv  (0KB)
      📄 cleaned_feature_engineered_bank_marketing.csv  (5480KB)
      📄 duration_summary.csv  (0KB)
      📄 expected_value_eur_base.csv  (1KB)
      📄 expected_value_usd_base.csv  (1KB)
      📄 job_subscription_summary.csv  (0KB)
      📄 job_summary.csv  (1KB)
      📄 poutcome_subscription_summary.csv  (0KB)
      📄 usd_eur_ranking_comparison.csv  (0KB)
    📁 notebook_02/
  📁 reports/
  📁 visuals/
